# Tokenization and embeddings

## 1. The intuition: how can a model understand words?

Imagine we give a Transformer this sentence:

> The teddy bear sat on the mat.

A neural network cannot do matrix multiplication on the word "teddy". It needs numbers.

The first two steps are:

![From text to vectors](images/01_text_to_vectors.svg)

The important distinction is:
- Tokenization decides how to split text into units and assign IDs.
- Embedding maps each token ID to a learned numerical vector.

Token IDs are just identifiers. Token 205 is not inherently more meaningful than token 101.

The embedding vectors, however, are learned during training and can capture useful relationships.

## 2. The mathematics

To keep the numbers small, we use the first three tokens of the sentence: "The teddy bear".

Suppose we have:
- Vocabulary size $V=6$
- Embedding dimension $d=4$
- Sequence length $T=3$

| Symbol | Name | Intuition | Example |
|---|---|---|---|
| $V$ | Vocabulary size | • How many unique tokens the model knows<br>• Fixed by the vocabulary | Our dictionary has bear, mat, on, sat, teddy, the, so $V = 6$. |
| $d$ | Embedding dimension | • How many learned numbers describe each token<br>• Fixed by the model | "teddy" is `[+0.7, +0.1, -0.3, +0.8]`, so $d = 4$.  |
| $T$ | Sequence length | • How many tokens are in the current input<br>• Changes with the input | "The teddy bear" has 3 tokens, so $T = 3$. |

**Key relationships**

| Matrix | Shape | Meaning |
|---|---|---|
| Embedding table $E$ | $V \times d = 6 \times 4$ | Stores a learned embedding for every token in the vocabulary. |
| Input embeddings $X$ | $T \times d = 3 \times 4$ | Contains the embeddings of the 3 tokens in our input sequence. |

![Embedding matrix E, the input sentence and the input embeddings X](images/02_embedding_matrix.svg)

**The lookup**

Token ID $i$ picks row $i$ of $E$:

$$x_i = E[i]$$

For "teddy" (ID 4): $x_4 = [+0.7,\ +0.1,\ -0.3,\ +0.8]$.

Stack one row per token to get the input matrix:

$$X = \begin{bmatrix} E[5] \\ E[4] \\ E[0] \end{bmatrix} = \begin{bmatrix} +0.1 & 0.0 & +0.3 & -0.1 \\ +0.7 & +0.1 & -0.3 & +0.8 \\ -0.1 & +0.9 & +0.4 & +0.2 \end{bmatrix} \in \mathbb{R}^{3 \times 4}$$

## 3. Reflection: why not just one-hot encoding?

#### 3.1 Their differences

A one-hot vector is a **name tag**. It says *which* word this is, and nothing else.

An embedding is a **position**. Words that are used in similar ways can end up at nearby positions.

| | One-hot | Learned embedding |
|---|---|---|
| Length | $V$ (vocabulary size) | $d$ (embedding dimension) |
| Contents | One 1, all others 0 | Dense, learned values |
| "teddy" (ID 4) | `[0, 0, 0, 0, 1, 0]` | `[+0.7, +0.1, -0.3, +0.8]` |
| "bear" (ID 0) | `[1, 0, 0, 0, 0, 0]` | `[-0.1, +0.9, +0.4, +0.2]` |
| Similarity | All distinct tokens are equally dissimilar | Similar tokens can have nearby vectors |
| Learned? | No, fixed by token ID | Yes, learned during training |

One-hot tells the model **who**. An embedding can also tell it **what it is like**.

#### 3.2 Their connection

**An embedding is essentially a learned transformation of a one-hot vector.**

Think of the one-hot vector $e_4$ as a panel of 6 switches. Only the "teddy" switch is on.

1. **Scale:** multiply each row of $E$ by its switch. An off switch (0) turns its row into zeros. The on switch (1) keeps its row.
2. **Add:** sum the rows. Only teddy's row is left, so the result is $E[4]$.

![Switches times E: scale each row, then add the rows](images/03_one_hot_lookup.svg)

In math, this is one boxed formula. With $e_i \in \mathbb{R}^{V}$ and $E \in \mathbb{R}^{V \times d}$:

$$\boxed{x_i = e_i^\top E}$$

The shapes are $(1 \times V)(V \times d) = (1 \times d)$. For "teddy", the zeros are the off switches:

$$x_4 = 0 \cdot E[0] + \dots + 1 \cdot E[4] + 0 \cdot E[5] = E[4]$$

So an embedding layer is a linear transformation of one-hot token vectors. This is what `torch.nn.Embedding` does conceptually. In practice it skips the zeros, reads row $i$ directly and never builds $e_i$.

## 4. In PyTorch: what does the embedding layer do, and not do?

**Goal:** check in code what the embedding layer does and what it cannot do.

- **Does:** a lookup that equals one-hot $\times E$, and a table that is learned
- **Cannot:** use context, or know word order

**Setup**

- Running sentence: "The teddy bear"
- IDs: `bear`=0, `mat`=1, `on`=2, `sat`=3, `teddy`=4, `the`=5
- The toy vocabulary ignores capitalization, so "The" and "the" are both ID 5
- A real model starts with a random table and learns it. Here we load our own numbers into `nn.Embedding`, so we can compare with the pictures

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Our embedding table E: one row per word, 4 numbers per row (V = 6, d = 4).
# The numbers are made up for illustration.
# A real model starts with random numbers and learns them during training.
E = torch.tensor([
    [-0.1, +0.9, +0.4, +0.2],  # row 0 = "bear"
    [+0.0, +0.4, -0.6, +0.3],  # row 1 = "mat"
    [+0.2, -0.5, +0.1, +0.6],  # row 2 = "on"
    [+0.5, -0.2, +0.2, -0.4],  # row 3 = "sat"
    [+0.7, +0.1, -0.3, +0.8],  # row 4 = "teddy"
    [+0.1, +0.0, +0.3, -0.1],  # row 5 = "the"
])

# Wrap E in an embedding layer. freeze=False means training is allowed to change the numbers.
embedding = nn.Embedding.from_pretrained(E, freeze=False)
print(embedding.weight.shape)          # (6, 4): 6 words, 4 numbers each
print(embedding.weight.requires_grad)  # True: the numbers can be learned

torch.Size([6, 4])
True


#### 4.1 It does: the lookup equals one-hot times $E$

Section 3 claimed $x_i = e_i^\top E$. Check it for "The teddy bear" (IDs 5, 4, 0):

In [2]:
ids = torch.tensor([5, 4, 0])  # token IDs of "The teddy bear"
X = embedding(ids)             # lookup: copy rows 5, 4 and 0 of E, shape (3, 4)

# The same thing as a matrix product: one row of 6 switches per token, then multiply by E
one_hot = F.one_hot(ids, num_classes=6).float()  # shape (3, 6)
print(one_hot)                                   # a single 1 in each row, at the token's ID
print(torch.allclose(one_hot @ embedding.weight, X))  # True: the product gives the same X

tensor([[0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0., 0.]])
True


#### 4.2 It does: the table is learned, and only the used rows move

- $E$ is a trainable weight matrix.
- To train it, we compute a loss from $X$ and back-propagate.
- A loss is one number that says how wrong the model is.
- We have no real task here, so `X.sum()` is a made-up loss, only there so we can compute gradients.
- The point is which rows receive a gradient.

In [3]:
# Use a fresh copy of E, so training here does not change our original table
trainable = nn.Embedding.from_pretrained(E.clone(), freeze=False)
X = trainable(torch.tensor([5, 4, 0]))  # look up "The teddy bear"

# A made-up loss, only here so we can compute gradients.
# backward() works out how each number in E would change the loss (its gradient)
X.sum().backward()

# For each row: did any of its numbers receive a gradient?
print((trainable.weight.grad != 0).any(dim=1))

with torch.no_grad():
    trainable.weight -= 0.1 * trainable.weight.grad  # one gradient step: nudge the numbers (learning rate 0.1)

# For each row: did any of its numbers change?
print((trainable.weight != E).any(dim=1))

tensor([ True, False, False, False,  True,  True])
tensor([ True, False, False, False,  True,  True])


**Only the rows of "The teddy bear" (0, 4 and 5) changed. Rows 1, 2 and 3 did not.**

**Why?** Only the words we looked up were used to compute the loss. Changing "mat" would not change the loss, so its gradient is 0 and it stays put.

Training only moves the words that appear.

**In a real model**

- The loss is next-word prediction, but the rule is the same: the lookup only sends gradients to the words that appeared.
- Other rows can still drift a little:
  - The output layer may reuse $E$ to score every word in the vocabulary.
  - Optimizer momentum keeps a row moving for a few steps after it appeared.

#### 4.3 It cannot: use context

- In "The teddy bear", "bear" is a toy.
- In "The bear sat on the mat", it could just as well be an animal.
- What vector does the layer return for "bear" in each sentence?

In [4]:
# Two sentences that both contain "bear" (ID 0)
the_teddy_bear = embedding(torch.tensor([5, 4, 0]))
the_bear_sat_on_the_mat = embedding(torch.tensor([5, 0, 3, 2, 5, 1]))

print(the_teddy_bear[2])           # the vector for "bear" in sentence 1 (position 2)
print(the_bear_sat_on_the_mat[1])  # the vector for "bear" in sentence 2 (position 1)
print(torch.equal(the_teddy_bear[2], the_bear_sat_on_the_mat[1]))  # True: identical

tensor([-0.1000,  0.9000,  0.4000,  0.2000], grad_fn=<SelectBackward0>)
tensor([-0.1000,  0.9000,  0.4000,  0.2000], grad_fn=<SelectBackward0>)
True


- The same row comes back whatever the neighbors are, so this is a **static** embedding.
- A toy bear and an animal bear get the same vector.
- Making the vector depend on the context is the job of **attention** (see the Attention notebook).

#### 4.4 It cannot: know word order

- Keep the same words and reverse their order.
- "The teddy bear" becomes "bear teddy The".
- Does the layer notice?

In [5]:
x1 = embedding(torch.tensor([5, 4, 0]))  # "The teddy bear"
x2 = embedding(torch.tensor([0, 4, 5]))  # "bear teddy The": same words, reversed

print(torch.equal(x2, x1.flip(0)))           # True: same rows, only the row order is reversed
print(torch.allclose(x1.sum(0), x2.sum(0)))  # True: adding up the rows ignores order, so the sums match

True
True


- The layer only returns rows. The rows themselves say nothing about where each word sat.
- A step that ignores row order sees the same input for both sentences.
- Word order has to be added separately, with **positional information** (see the Transformer building blocks notebook).

## 5. Tokenization: what decides $T$?

- The tokenizer cuts the text into tokens. The number of tokens is $T$.
- Each token becomes one row of $X$, and takes one position in the model's context.
- Cut the same text more finely and $T$ grows, so $X$ gets more rows.

![The same text split into 3 word tokens or 14 character tokens](images/04_tokenization_sets_T.svg)

Real tokenizers sit in between: they split text into pieces of words (subwords). Here is the same effect in code, with two toy tokenizers:

In [6]:
text = "The teddy bear"

word_tokens = text.split()  # ["The", "teddy", "bear"]
char_tokens = list(text)    # ["T", "h", "e", " ", "t", ...]

def embed(tokens, d=4):
    vocab = {tok: i for i, tok in enumerate(sorted(set(tokens)))}  # token -> ID
    ids = torch.tensor([vocab[tok] for tok in tokens])
    table = nn.Embedding(len(vocab), d)  # toy table with random numbers
    return table(ids)                    # X, one row per token

print(embed(word_tokens).shape)  # (T, d) with word tokens
print(embed(char_tokens).shape)  # (T, d) with character tokens

torch.Size([3, 4])
torch.Size([14, 4])


- The same text gave $T = 3$ or $T = 14$, so $X$ had 3 or 14 rows.
- A model's context window is the maximum number of tokens it can read at once. It counts tokens, not words, so twice as many tokens uses twice as many context positions.
- So the tokenizer decides how much of the context a text uses.